# 第5章：SFT 指令微调 (Supervised Fine-Tuning)

## 本章目标
- 理解 instruction tuning 的数据格式（Alpaca format）
- 理解 loss masking：只在 response 部分计算 loss
- 使用 LoRA 高效微调大模型
- 使用 trl 的 SFTTrainer 完成完整的 SFT 流程

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install torch transformers trl peft datasets accelerate bitsandbytes
else:
    print("本地环境运行，请确保已按 intro.md 配置好环境")

## SFT 速览

Pretrained model 只会"续写文本"。SFT 教会模型"听指令"。

核心变化：
- 数据格式：从纯文本变成 (instruction, response) 对
- Loss masking：只在 response 部分计算 loss（不教模型学指令本身）
- LoRA：只训练少量参数，而不是全量微调

参考：[InstructGPT](https://arxiv.org/abs/2203.02155), [LoRA](https://arxiv.org/abs/2106.09685)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("tatsu-lab/alpaca", split="train[:5000]")

def format_alpaca(example):
    if example["input"]:
        prompt = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{example['instruction']}\n\n### Response:\n"
    return {"text": prompt + example["output"]}

dataset = dataset.map(format_alpaca)
print(f"数据集大小: {len(dataset)}")
print(f"样本示例:\n{dataset[0]['text'][:300]}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
import torch

model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto"
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./sft_output",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    max_seq_length=512,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)
trainer.train()
print("SFT 训练完成")

In [ ]:
def test_sft_model(model, tokenizer, instruction):
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

test_cases = [
    "Explain what a transformer is in simple terms.",
    "Write a haiku about programming.",
    "What is the capital of France?",
]
for tc in test_cases:
    print(f"Q: {tc}")
    print(f"A: {test_sft_model(model, tokenizer, tc)}\n")

## Loss Masking 原理

SFT 的关键细节：**只在 response 部分计算 loss**。

为什么不全程计算 loss？因为 instruction 部分是"问题"，我们不想让模型学会生成问题，只想让它学会生成回答。

实现方式：创建一个 mask tensor，instruction 部分为 0，response 部分为 1，loss 乘以 mask 后求平均。

In [ ]:
import torch
import torch.nn.functional as F

def compute_loss_with_mask(logits, labels, response_start_positions):
    """只在 response 部分计算 loss"""
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
    loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    mask = torch.zeros_like(shift_labels, dtype=torch.bool)
    for i, start in enumerate(response_start_positions):
        mask[i, start:] = True
    masked_loss = loss.view(shift_labels.shape) * mask
    return masked_loss.sum() / mask.sum()

# 示例
logits = torch.randn(2, 10, 100)  # batch=2, seq_len=10, vocab=100
labels = torch.randint(0, 100, (2, 10))
response_starts = [3, 5]  # response 从第 3/5 个 token 开始
loss = compute_loss_with_mask(logits, labels, response_starts)
print(f"Masked loss: {loss.item():.4f}")
print("只有 response 部分的 token 参与了 loss 计算。")

## 练习

1. 修改 LoRA 的 rank (r=8, r=32, r=64)，对比训练速度和效果
2. 尝试不同的 target_modules（加入 k_proj, v_proj, o_proj）
3. 用自己的 instruction dataset 替换 Alpaca 数据

## 延伸阅读

- [LoRA 论文](https://arxiv.org/abs/2106.09685)
- [trl SFTTrainer 文档](https://huggingface.co/docs/trl/sft_trainer)
- [Alpaca 项目](https://github.com/tatsu-lab/stanford_alpaca)